In [1]:
%load_ext cudf.pandas

In [2]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/home/azureuser/cloudfiles/data/tabpfn-3"

In [13]:
import pathlib
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

COMP_DIR = pathlib.Path("/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-agils-ff-gpu/code/Users/agils/playground-series-s6e6/data")

train = pd.read_csv(COMP_DIR / "train.csv", index_col="id")
test = pd.read_csv(COMP_DIR / "test.csv", index_col="id")
sample_submission = pd.read_csv(COMP_DIR / "sample_submission.csv")

train.columns = train.columns.str.lower()
test.columns = test.columns.str.lower()

print(train.shape, test.shape)

(577347, 11) (247435, 10)


In [14]:
train.head(5)

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
id,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [5]:
ohe_df = pd.get_dummies(train[["spectral_type", "galaxy_population"]], drop_first=True, dtype="int")
ohe_df.columns = ohe_df.columns.str.lower()

train = pd.concat([train, ohe_df], axis=1)
train = train.drop(columns=["spectral_type", "galaxy_population"])

In [6]:
ohe_df_test = pd.get_dummies(test[["spectral_type", "galaxy_population"]], drop_first=True, dtype="int")
ohe_df_test.columns = ohe_df_test.columns.str.lower()

test = pd.concat([test, ohe_df_test], axis=1)
test = test.drop(columns=["spectral_type", "galaxy_population"])

In [152]:
import numpy as np
from itertools import combinations

def do_feature_engineering(df):
    df = df.copy()

    magnitudes = ["u", "g", "r", "i", "z"]

    for c1, c2 in combinations(magnitudes, 2):
        df[f"{c1}-{c2}"] = df[c1] - df[c2]

    magnitudes_combos = [
        f"{a}-{b}" for a, b in combinations(magnitudes, 2)
    ]

    for c1, c2 in combinations(magnitudes_combos, 2):
        df[f"{c1}_x_{c2}"] = df[c1] * df[c2]
        df[f"{c1}_x_{c2}_x_rs"] = df[f"{c1}_x_{c2}"] * df["redshift"]

    df["is_very_low_z"] = (df["redshift"] < 0.05).astype('float32')
    df["is_high_z"] = (df["redshift"] > 1.5).astype('float32')

    df = df.drop(columns=magnitudes)
    
    return df

train = do_feature_engineering(train)
test = do_feature_engineering(test)

In [17]:
def add_floor_categories(train, test):

    mapping = {}

    for num_col in ["u","g","r","i","z","redshift","alpha","delta"]:

        cat_col = f"{num_col}_floor_cat"

        floored_train = np.floor(train[num_col])
        codes_train, uniques = pd.factorize(floored_train, sort=True)

        mapping[num_col] = {val:i for i, val in enumerate(uniques)}

        train[cat_col] = codes_train.astype("int32")

        floored_test = np.floor(test[num_col])
        codes_test = floored_test.map(mapping[num_col]).fillna(-1).astype("int32")
        test[cat_col] = codes_test

    return train, test

train, test = add_floor_categories(train, test)

In [15]:
TARGET = "class"
NUM_COLS = [c for c in train.select_dtypes(include=["number"]).columns]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + [TARGET]]

In [16]:
CAT_COLS

['spectral_type', 'galaxy_population']

In [17]:
for cat_col in CAT_COLS:
    train[cat_col] = train[cat_col].astype("category")
    test[cat_col] = test[cat_col].astype("category")

In [18]:
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}

train[TARGET] = train[TARGET].map(TARGET_MAPPING)

In [22]:
import torch

device = "gpu" if torch.cuda.is_available() else "cpu"

XGB_PARAMS = {
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "num_class": 3,
    "tree_method": "hist",
    "device": device,
    "n_estimators": 10000,
    "learning_rate": 0.01,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "random_state": 42,
    "enable_categorical": True
}

CAT_PARAMS = {
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "iterations": 20000,
    "learning_rate": 0.01,
    "depth": 6,
    "min_data_in_leaf": 10,
    "l2_leaf_reg": 1.0,
    "random_seed": 42,
    "task_type": "GPU",
    "early_stopping_rounds": 200,
    "verbose": 0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "use_best_model": True

}


REALMLP_PARAMS = {
    "device": "cuda",
    "random_state": 42,
    "verbosity": 1,
    "val_metric_name": "1-balanced_accuracy",
    "n_cv": 1,
    "n_refit": 0,
    "use_ls": False,
    "batch_size": 4096,
    "predict_batch_size": 8192,
    "n_threads": 4,
    "use_plr_embeddings": True,
    "lr": 0.01,
    "n_epochs": 1024,
    "use_early_stopping": True,
    "early_stopping_additive_patience": 50,
    "early_stopping_multiplicative_patience": 2
}


In [194]:
FLOOR_CAT_COLS = [c for c in train.columns if c.endswith("floor_cat")]

In [23]:
import numpy as np
import logging

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import TargetEncoder
from tabpfn import TabPFNClassifier
from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier
from autogluon.tabular import TabularPredictor

logging.getLogger("tabpfn").setLevel(logging.DEBUG)

MODEL_IN_USE = 'XGB'

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

X = train[NUM_COLS + CAT_COLS]
y = train[TARGET]

X_test = test[NUM_COLS + CAT_COLS]

oof_probs = np.zeros((X.shape[0], 3))
tst_probs = np.zeros((X_test.shape[0], 3))

feat_importance_df = pd.DataFrame(index=NUM_COLS + CAT_COLS)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    if MODEL_IN_USE == 'XGB':

        print('Using XGB Model')

        model = XGBClassifier(
            **XGB_PARAMS
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            sample_weight=train_weights,
            verbose=0
        )

        feat_importance_df[f"fold_{fold}"] = model.feature_importances_

    if MODEL_IN_USE == 'CATBOOST':

        print('Using CatBoost Model')


        model = CatBoostClassifier(**CAT_PARAMS)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            sample_weight=train_weights,
            cat_features=CAT_COLS,
            verbose=100
        )

    if MODEL_IN_USE == 'REALMLP':

        print('Using RealMLP Model')

        model = RealMLP_TD_Classifier(
            **REALMLP_PARAMS
        )

        model.fit(
            X_train, y_train,
            X_val=X_val, y_val=y_val
        )
        
    oof_probs[val_idx] = model.predict_proba(X_val)
    tst_probs += model.predict_proba(X_test)

    oof_score = balanced_accuracy_score(y_val, np.argmax(oof_probs[val_idx], axis=1))
    print(f'Fold {fold+1} Balanced Accuracy: {oof_score:.5f}')

tst_probs /= 5

oof_score_full = balanced_accuracy_score(y, np.argmax(oof_probs, axis=1))

print(f"Full OOF balanced_accuracy: {oof_score_full:.5f}")

Using XGB Model
Fold 1 Balanced Accuracy: 0.96551
Using XGB Model
Fold 2 Balanced Accuracy: 0.96507
Using XGB Model
Fold 3 Balanced Accuracy: 0.96505
Using XGB Model
Fold 4 Balanced Accuracy: 0.96495
Using XGB Model
Fold 5 Balanced Accuracy: 0.96500
Full OOF balanced_accuracy: 0.96512


In [198]:
oof_df = pd.DataFrame(
    oof_probs, columns=['p1', 'p2', 'p3']
)

oof_df['class'] = y.values

oof_df.to_parquet('/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-agils-ff-gpu/code/Users/agils/playground-series-s6e6/submissions/oof_cat.parquet', index=False)

In [199]:
tst_df = pd.DataFrame(
    tst_probs, columns=['p1', 'p2', 'p3']
)

tst_df.to_parquet('/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-agils-ff-gpu/code/Users/agils/playground-series-s6e6/submissions/oof_test_cat.parquet', index=False)

In [200]:
from scipy.optimize import minimize, differential_evolution

init_weights = np.ones(3)
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]

def threshold_objective(weights, oof_probs, y_true):

    weights = weights / (np.sum(weights) + 1e-12)

    scaled_probs = oof_probs * weights
    preds = np.argmax(scaled_probs, axis=1)
    score = balanced_accuracy_score(y_true, preds)

    return -score

threshold_result = differential_evolution(
    threshold_objective,
    bounds=bounds,
    args=(oof_probs, y.to_numpy()),
    seed=42,
    maxiter=1000,
    popsize=20,
    polish=False
)


In [201]:
best_weights = threshold_result.x
best_weights = best_weights / best_weights.sum()
best_weights

array([0.31673594, 0.3595279 , 0.32373616])

In [202]:
-threshold_result.fun

np.float64(0.9621694124320442)

In [203]:
TARGET_MAPPING_INV = {
    0: "GALAXY",
    1: "QSO",
    2: "STAR"
}

In [204]:
SUBMISSION_PATH = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-agils-ff-gpu/code/Users/agils/playground-series-s6e6/submissions"
SUBMISSION_PATH = pathlib.Path(SUBMISSION_PATH)

sample_submission['class'] = np.argmax(tst_probs * best_weights, axis=1)
sample_submission['class'] = sample_submission['class'].map(TARGET_MAPPING_INV)
sample_submission.to_csv(SUBMISSION_PATH / 'submission_cat_9621694124320442.csv', index=False)

In [205]:
sample_submission

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
...,...,...
247430,824777,QSO
247431,824778,QSO
247432,824779,GALAXY
247433,824780,QSO
